In [175]:
import polars as pl 
from thefuzz import fuzz
import json 
import copy
import re

df = pl.read_csv("./output/dictionary_senses.csv")


def clean(definition):
    definition = definition.replace("[colloquial]", "") \
        .replace("[written]", "") \
        .replace("written and colloquial]", "") \
        .replace("(Cantonese)", "") \
        .replace("(Cant.)", "") \
        .replace("(spoken)", "")
    
    definition = re.sub(r"\b(verb|adverb|adjective|noun|pronoun|slang|to)\b", "", definition)
    definition = definition.replace("(", "").replace(")", "").replace("[", "").replace("]", "")
    return definition

# Put each definition in its own row and then dedupe
new_rows = []
for row in df.iter_rows(named=True):
    definition_string = row["definition"]
    if ";" in definition_string:
        definitions = definition_string.split(";")
        
        for definition in definitions:
            definition = clean(definition)
            definition = definition.lower().strip()
            if definition:
                copied_row = copy.deepcopy(row)
                copied_row["definition"] = definition
                new_rows.append(copied_row)
    else:
        definition = clean(definition_string)
        definition = definition.lower().strip()
        if definition:
            row["definition"] = definition
            new_rows.append(row)
        
clean_df = pl.DataFrame(new_rows).unique()

In [176]:
from sentence_transformers import SentenceTransformer


model = SentenceTransformer("whaleloops/phrase-bert")

filtered_rows = []

for trad_word, group in clean_df.group_by("traditional"):
    defs = group.select("definition").to_series()
    
    if len(defs) <= 1:
        # If only one definition, keep it
        filtered_rows.extend(group.to_dicts())
        continue
    
    def_embeddings = model.encode(defs)
    
    # Build similarity matrix
    similarity_scores = {}
    for def1_index in range(len(defs)):
        for def2_index in range(def1_index + 1, len(defs)):
            def1, def2 = defs[def1_index], defs[def2_index]
            
            # Calculate similarity score (same logic as your original code)
            num_def1_unique = len(set(def1.split()) - set(def2.split()))
            num_def2_unique = len(set(def2.split()) - set(def1.split()))
            total_unique = num_def1_unique + num_def2_unique

            semantic_sim = model.similarity(def_embeddings[def1_index], 
                                          def_embeddings[def2_index]).item()
            
            ratio = fuzz.token_sort_ratio(def1, def2)
            score = (0.5 * ratio) + (0.5 * 100 * semantic_sim) - (1.5 * total_unique)
            
            similarity_scores[(def1_index, def2_index)] = score
    
    # Greedy selection: keep definitions that don't conflict with already selected ones
    selected_indices = []
    remaining_indices = list(range(len(defs)))
    
    while remaining_indices:
        # Pick the first remaining definition
        current_idx = remaining_indices.pop(0)
        selected_indices.append(current_idx)
        
        # Remove any remaining definitions that are too similar to the selected one
        to_remove = []
        for remaining_idx in remaining_indices:
            pair = (min(current_idx, remaining_idx), max(current_idx, remaining_idx))
            if pair in similarity_scores and similarity_scores[pair] >= 50:
                to_remove.append(remaining_idx)
        
        for idx in to_remove:
            remaining_indices.remove(idx)
    
    # Add the selected rows to our filtered list
    group_rows = group.to_dicts()
    for idx in selected_indices:
        filtered_rows.append(group_rows[idx])

# Create new dataframe with filtered rows
filtered_df = pl.DataFrame(filtered_rows)

In [177]:
filtered_df = filtered_df.filter(
    pl.col("traditional").count().over("traditional") > 1
)
filtered_df.write_csv("./output/filtered_dictionary_senses.csv")

In [178]:
# NOTE: Tuned thresholds using "modal particle" and "skinny" as examples